COMPARATIVE EVALUATION OF SCALABLE MULTI-CLASS CLASSIFICATION MODELS FOR NYPD ARREST SEVERITY PREDICTION USING DISTRIBUTED PYSPARK AND TABLEAU ANALYTICS

**1) Data Acquisition & Environment Setup**

**Cell 1: Spark Installation**

In [1]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/universe/o/openjdk-8/openjdk-8-jre-headless_8u472-ga-1%7e22.04_amd64.deb  404  Not Found [IP: 91.189.92.23 80]
E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/universe/o/openjdk-8/openjdk-8-jdk-headless_8u472-ga-1%7e22.04_amd64.deb  404  Not Found [IP: 91.189.92.23 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?


In [2]:
!java -version

openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-122.04, mixed mode, sharing)


**Cell 2: Pip Instll Pyspark**

In [4]:
!pip install pyspark==3.5.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 19.3 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488493 sha256=d5e766ce2a3641b1d1ffe34936e06f844e522b3fb0f2ba919635389ff63b065f
  Stored in directory: /root/.cache/pip/wheels/b1/91/5f/283b53010a8016a4ff1c4a1edd99bbe73afacb099645b5471b
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.2
    Uninstalling pyspark-4.0.2:
      Successfully uninstalled pyspark-4.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-conn

In [5]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/usr/local/lib/python3.10/dist-packages/pyspark"

**CELL 3** Start SparkSession (Big Data tuning)

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("NYC_Cleaning")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.network.timeout", "600s")
    .config("spark.executor.heartbeatInterval", "60s")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 3.5.1


**CELL 4**  Upload again (confirm correct filename)


In [ ]:
from google.colab import files
uploaded = files.upload()
print(uploaded.keys())

Saving NYPD_Arrests_Data__Historic_.csv to NYPD_Arrests_Data__Historic_.csv
dict_keys(['NYPD_Arrests_Data__Historic_.csv'])


**CELL 5**  Confirm file exists in /content



In [ ]:
!ls -lh /content | sed -n '1,200p'

total 1.2G
-rw-r--r-- 1 root root 1.2G Feb  9 18:28 NYPD_Arrests_Data__Historic_.csv
drwxr-xr-x 1 root root 4.0K Jan 16 14:24 sample_data


**CELL 6**  **Robust file path detection**



In [ ]:
import os, glob

# Most likely exact path
CSV_PATH = "/content/NYPD_Arrests_Data_Historic_.csv"

# If it uploaded with a slightly different name, auto-pick a matching file
if not os.path.exists(CSV_PATH):
    matches = glob.glob("/content/*NYPD*Arrests*Historic*.csv")
    print("Matches found:", matches)
    if matches:
        CSV_PATH = matches[0]

print("Using CSV_PATH:", CSV_PATH)
print("Exists?", os.path.exists(CSV_PATH))

Matches found: ['/content/NYPD_Arrests_Data__Historic_.csv']
Using CSV_PATH: /content/NYPD_Arrests_Data__Historic_.csv
Exists? True


**CELL 7**  Install PySpark again (safe repeat)



In [ ]:
!pip -q install pyspark==3.5.1

**CELL 8** Start SparkSession again (stable runtime)



In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("NYPD_Arrests_Cleaning")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.network.timeout", "600s")
    .config("spark.executor.heartbeatInterval", "60s")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 3.5.1


**2) Data Ingestion + Initial Profiling**